In [0]:
from pyspark.sql.types import StringType, IntegerType, DateType, BooleanType
import pyspark.sql.functions as F
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("catalog_name", "ecommerce", "Catalog Name")
dbutils.widgets.text("storage_account_name", "stgecommerceeastus001", "Storage Account Name")
dbutils.widgets.text("container_name", "ecomm-raw-data", "Container Name")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
container_name = dbutils.widgets.get("container_name")

In [0]:
df = spark.readStream \
.format("delta") \
.table(f"{catalog_name}.bronze.brz_order_items")


In [0]:
df = df.dropDuplicates(['order_id', 'item_seq'])

In [0]:
df = df.withColumn(
    "quantity",
    F.when(F.col('quantity') == "Two", 2)
    .otherwise(F.col('quantity'))
    .cast('int')          
)



In [0]:
df = df.withColumn(
    "unit_price",
    F.regexp_replace('unit_price', "[$]", "")
    .cast('double')
)


In [0]:
df = df.withColumn(
    "discount_pct",
    F.regexp_replace('discount_pct', "[%]", "")
    .cast('double')
)

In [0]:
df = df.withColumn(
    'coupon_code',
    F.lower(F.trim(f.col('coupon_code')))
)

In [0]:
df = df.withColumn('channel', 
    F.when(F.col('channel') == 'web', 'Website')
    .when(F.col('channel') == 'app', 'Mobile')
    .otherwise(F.col('channel'))
)

In [0]:
df = df.withColumn(
    "processed_time",
    F.current_timestamp()
)

In [0]:
silver_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/silver/fact_order_items/"
print(silver_checkpoint_path)

In [0]:
def upsert_to_silver(moicroBatchDF, batchId):
    table_name = f"{catalog_name}.silver.slv_order_items"
    if not spark.catalog.tableExists(table_name):
        moicroBatchDF.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(table_name)
        spark.sql(
            f"""
            ALTER TABLE {table_name} SET TBLPROPERTIES (
                delta.enableChangeDataFeed = true
            )
            """
        )    
    else:
        deltaTable = DeltaTable.forName(spark, table_name)
        deltaTable.alias("silver_table").merge(
            moicroBatchDF.alias("batch_table"),
            "silver_table.order_id = batch_table.order_id AND silver_table.item_seq = batch_table.item_seq",
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute() 
        

    

In [0]:
df.writeStream.trigger(availableNow=True).foreachBatch(
    upsert_to_silver
).option("checkpointLocation", silver_checkpoint_path).option(
    "mergeSchema", "true"
).outputMode(
    "update"
).start().awaitTermination()